# Baseline and Linear Regression analysis
Bekzat Gulzhigit uulu · DSC 345 · November 12, 2026

Compare a training-calibrated proportional formula, usage-only Linear Regression, and expanded Linear Regression. A training-mean prediction is included as a simple reference. These results are a contribution for group integration; other members' models are not included.

The source CSV and results are included. Run cells from the repository root with the packages in `requirements.txt` installed. No network access is required.

In [1]:
from pathlib import Path
import json
import pandas as pd
from src.run_baselines import load_data, model_specs, main, TARGET, FEATURES
ROOT = Path.cwd()
assert (ROOT / "data/Steel_industry_data.csv").exists(), "Open the notebook from the project folder."


## 1. Inspect the source observations
The target is `CO2(tCO2)`. Dates are parsed literally as day/month/year and sorted. The original CSV places midnight after 23:45 within each date; this analysis does not infer a date correction. Missing values and duplicates cause an error so they cannot silently alter the shared split.

In [2]:
data = load_data(ROOT / "data/Steel_industry_data.csv")
print("Rows:", len(data))
print("Missing values:", int(data.isna().sum().sum()))
print("Target values:", sorted(data[TARGET].unique().tolist()))
print(data[["source_row", "date", "Usage_kWh", TARGET]].head().to_string(index=False))


Rows: 35040
Missing values: 0
Target values: [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07]
 source_row                date  Usage_kWh  CO2(tCO2)
         95 2018-01-01 00:00:00       3.42        0.0
          0 2018-01-01 00:15:00       3.17        0.0
          1 2018-01-01 00:30:00       4.00        0.0
          2 2018-01-01 00:45:00       3.24        0.0
          3 2018-01-01 01:00:00       3.31        0.0


## 2. Define the chronological split
Train on the earliest 70%, test on the latest 30%. Every teammate should reuse the source row IDs in `results/split_manifest.csv`. The target is never included in the inputs. `WeekStatus` is redundant with `Day_of_week`, so it is excluded from the expanded model.

In [3]:
cut = len(data) * 7 // 10
train, test = data.iloc[:cut], data.iloc[cut:]
print("Training rows:", len(train), "from", train.date.min(), "to", train.date.max())
print("Test rows:", len(test), "from", test.date.min(), "to", test.date.max())
print("Expanded inputs:", FEATURES)
assert train.date.max() < test.date.min()
assert not set(train.source_row).intersection(test.source_row)


Training rows: 24528 from 2018-01-01 00:00:00 to 2018-09-13 11:45:00
Test rows: 10512 from 2018-09-13 12:00:00 to 2018-12-31 23:45:00
Expanded inputs: ['Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor', 'NSM', 'Day_of_week', 'Load_Type']


## 3. Understand and fit the models
The proportional baseline forces the fitted line through zero. Usage-only Linear Regression learns both slope and intercept. Expanded Linear Regression adds other measurements. Its numeric scaling and category encoding are fitted only on training data.

`main()` runs three expanding validation folds inside the training period, fits each model on all training rows, evaluates the fixed test rows, and saves predictions, metrics, and figures. No hyperparameters are tuned. The full implementation is in `src/run_baselines.py`.

In [4]:
for name, (features, estimator) in model_specs().items():
    print(name, "—", len(features), "raw inputs;", type(estimator).__name__)
summary, metrics = main()


mean_reference — 1 raw inputs; DummyRegressor
proportional_baseline — 1 raw inputs; LinearRegression
linear_usage — 1 raw inputs; LinearRegression
linear_expanded — 8 raw inputs; Pipeline
                model partition     n      MAE     RMSE        R2  negative_predictions
       mean_reference      test 10512 0.013528 0.015138 -0.019650                     0
proportional_baseline      test 10512 0.001993 0.002247  0.977527                     0
         linear_usage      test 10512 0.001155 0.001837  0.984986                  2291
      linear_expanded      test 10512 0.001251 0.001852  0.984730                  3052
Training-calibrated proportional factor: 0.0004543326
Usage-only formula: 0.0004757292 * Usage_kWh + (-0.0014959152)


## 4. Compare errors and check limitations
MAE and RMSE are in native target units, and lower is better. R² measures fit relative to predicting the test mean; it is not percentage accuracy. UCI's column name and unit description conflict, so do not convert these numbers into kilograms or tonnes.

Usage-only Linear Regression has the lowest test errors in this comparison. Additional features slightly worsen these linear results. This does not establish the winner against Random Forest or Gradient Boosting.

In [5]:
validation = pd.read_csv(ROOT / "results/chronological_validation.csv")
print("Average training-period validation metrics:")
print(validation.groupby("model")[["MAE", "RMSE", "R2"]].mean().to_string())
print("\nTest predictions below zero:")
print(metrics.loc[metrics.partition == "test", ["model", "negative_predictions"]].to_string(index=False))
print("\nProportional coefficient:", summary["proportional_factor"])
print("Usage-only slope:", summary["usage_slope"], "intercept:", summary["usage_intercept"])


Average training-period validation metrics:
                            MAE      RMSE        R2
model                                              
linear_expanded        0.001496  0.002065  0.982195
linear_usage           0.001211  0.001973  0.983723
mean_reference         0.014299  0.015808 -0.044048
proportional_baseline  0.001988  0.002277  0.978333

Test predictions below zero:
                model  negative_predictions
       mean_reference                     0
proportional_baseline                     0
         linear_usage                  2291
      linear_expanded                  3052

Proportional coefficient: 0.0004543325652058064
Usage-only slope: 0.0004757292404148052 intercept: -0.0014959151734035784


## 5. Presentation figures
Metrics use individual 15-minute observations. The daily-mean plot is only a visualization.

![Test error comparison](figures/test_errors.png)

![Actual and predicted target values](figures/actual_vs_predicted.png)

![Daily mean predictions](figures/test_daily_means.png)

## What to say
Electricity usage alone provides a strong baseline. More inputs did not improve this linear comparison. The target is coarse, negative linear predictions need attention, and a high R² in one facility does not prove reliable forecasting elsewhere. See `results/README.md` for methodology, interpretation, and limitations.

## Source
V E, S., Shin, C., & Cho, Y. (2021). *Steel Industry Energy Consumption*. UCI Machine Learning Repository. https://doi.org/10.24432/C52G8C . The included original CSV is distributed under CC BY 4.0. Derived outputs are separate.